# Week 9: Two-Stage ECE Model Training
## Emotion-Cause-Extraction with RoBERTa (Days 5-8)

**Objective:** Train a two-stage ECE model using RoBERTa-base to extract emotion causes from text.

**Model Architecture:**
- **Base:** RoBERTa-base (pretrained transformer)
- **Head 1:** Clause Classifier (binary: contains cause or not)
- **Head 2:** Span Extractor (BIO tagging for cause boundaries)

**Training Strategy:**
- Combined loss: `0.3 * clause_loss + 0.7 * span_loss`
- Evaluation metric: F1 score (seqeval)
- Logging: Weights & Biases (wandb)

**Author:** Aura ML Project  
**Date:** November 16, 2025

---

## 📦 Install Required Libraries

Install all necessary libraries for training the ECE model.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install transformers datasets
%pip install seqeval
%pip install wandb
%pip install scikit-learn
%pip install tqdm

## 🔧 Import Libraries and Setup

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import wandb

# Transformers
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaPreTrainedModel,
    RobertaConfig,
    get_linear_schedule_with_warmup
)

# Evaluation
from seqeval.metrics import f1_score, classification_report
from sklearn.metrics import accuracy_score

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

# Check if CUDA is available
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("ℹ️  Running on CPU")

---
## 📂 Part 1: Data Preprocessing & Setup

### 1.1 Load ECE Dataset

Load the preprocessed ECE dataset from JSON files created in the previous pipeline.

In [ ]:
# Define paths
DATA_DIR = Path("ece_dataset")
TRAIN_FILE = DATA_DIR / "ece_train.json"
VAL_FILE = DATA_DIR / "ece_val.json"
TEST_FILE = DATA_DIR / "ece_test.json"

def load_ece_data(file_path: Path) -> List[Dict]:
    """Load ECE dataset from JSON file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# Load all splits
print("Loading ECE dataset...")
train_data = load_ece_data(TRAIN_FILE)
val_data = load_ece_data(VAL_FILE)
test_data = load_ece_data(TEST_FILE)

print(f"\n{'='*60}")
print(f"📊 ECE DATASET STATISTICS")
print(f"{'='*60}")
print(f"Training samples:   {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"Test samples:       {len(test_data):,}")
print(f"{'='*60}")
print(f"Total samples:      {len(train_data) + len(val_data) + len(test_data):,}")
print(f"{'='*60}\n")

# Display sample
print("Sample from training set:")
sample = train_data[0]
print(f"Text:     {sample['text']}")
print(f"Emotion:  {sample['emotion']}")
print(f"Cause:    {sample['cause']}")
print(f"Source:   {sample['source']}")
print(f"\n✅ Data loaded successfully!")

### 1.2 Load RoBERTa Tokenizer

Load the RoBERTa-base tokenizer for text preprocessing.

In [ ]:
# Load RoBERTa tokenizer
MODEL_NAME = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

print(f"✅ Loaded tokenizer: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Max length: {tokenizer.model_max_length}")

# Test tokenization
test_text = "I'm anxious because I have exams next week"
tokens = tokenizer.tokenize(test_text)
print(f"\nTest tokenization:")
print(f"Text: {test_text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {tokenizer.convert_tokens_to_ids(tokens)}")

### 1.3 BIO Tagging & Tokenization

Create BIO tags (B-Cause, I-Cause, O) for each token to identify cause spans.

In [ ]:
# Define BIO label mappings
BIO_LABELS = ["O", "B-Cause", "I-Cause"]
LABEL2ID = {label: idx for idx, label in enumerate(BIO_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

print(f"BIO Labels: {BIO_LABELS}")
print(f"Label to ID mapping: {LABEL2ID}")

def create_bio_tags(text: str, cause: str, tokens: List[str]) -> List[str]:
    """
    Create BIO tags for tokens based on cause substring.
    
    Args:
        text: Original text
        cause: Cause substring
        tokens: List of tokens from tokenizer
        
    Returns:
        List of BIO tags for each token
    """
    # Initialize all tags as 'O' (Outside)
    bio_tags = ['O'] * len(tokens)
    
    # Find cause in text (case-insensitive)
    text_lower = text.lower()
    cause_lower = cause.lower()
    
    # Find the start position of cause in text
    cause_start_char = text_lower.find(cause_lower)
    
    if cause_start_char == -1:
        # Cause not found as substring (fallback case), return all 'O'
        return bio_tags
    
    cause_end_char = cause_start_char + len(cause)
    
    # Tokenize and track character positions
    current_pos = 0
    in_cause = False
    
    for idx, token in enumerate(tokens):
        # Remove Ġ prefix (RoBERTa's space marker)
        token_clean = token.replace('Ġ', ' ').strip()
        
        # Find token position in original text
        token_start = text_lower.find(token_clean.lower(), current_pos)
        
        if token_start == -1:
            current_pos += 1
            continue
            
        token_end = token_start + len(token_clean)
        
        # Check if token overlaps with cause span
        if token_start >= cause_start_char and token_start < cause_end_char:
            if not in_cause:
                bio_tags[idx] = 'B-Cause'
                in_cause = True
            else:
                bio_tags[idx] = 'I-Cause'
        elif token_end > cause_start_char and token_end <= cause_end_char:
            if not in_cause:
                bio_tags[idx] = 'B-Cause'
                in_cause = True
            else:
                bio_tags[idx] = 'I-Cause'
        else:
            in_cause = False
        
        current_pos = token_end
    
    return bio_tags


# Test BIO tagging
test_sample = train_data[0]
test_tokens = tokenizer.tokenize(test_sample['text'])
test_bio = create_bio_tags(test_sample['text'], test_sample['cause'], test_tokens)

print(f"\n{'='*80}")
print(f"BIO TAGGING TEST")
print(f"{'='*80}")
print(f"Text:  {test_sample['text']}")
print(f"Cause: {test_sample['cause']}")
print(f"\nTokens and BIO tags:")
for token, bio in zip(test_tokens, test_bio):
    print(f"  {token:20s} -> {bio}")
print(f"{'='*80}")

### 1.4 Create PyTorch Dataset

Define a custom PyTorch Dataset class for ECE data with tokenization and BIO tagging.

In [ ]:
class ECEDataset(Dataset):
    """PyTorch Dataset for Emotion-Cause-Extraction."""
    
    def __init__(self, data: List[Dict], tokenizer: RobertaTokenizer, max_length: int = 128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        cause = item['cause']
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Get tokens for BIO tagging
        tokens = self.tokenizer.tokenize(text)
        
        # Create BIO tags
        bio_tags = create_bio_tags(text, cause, tokens)
        
        # Convert to IDs and pad to max_length
        bio_ids = [LABEL2ID[tag] for tag in bio_tags]
        
        # Add padding for special tokens and max_length
        # RoBERTa uses <s> and </s> tokens
        bio_ids = [LABEL2ID['O']] + bio_ids + [LABEL2ID['O']]  # Add O for <s> and </s>
        
        # Pad to max_length
        if len(bio_ids) < self.max_length:
            bio_ids += [LABEL2ID['O']] * (self.max_length - len(bio_ids))
        else:
            bio_ids = bio_ids[:self.max_length]
        
        # Clause label: 1 if cause exists in text, 0 otherwise
        # For keyword-based extraction, definitely has cause
        # For fallback, the whole text is the cause
        clause_label = 1 if cause.strip() else 0
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'bio_labels': torch.tensor(bio_ids, dtype=torch.long),
            'clause_label': torch.tensor(clause_label, dtype=torch.float)
        }


# Create datasets
MAX_LENGTH = 128

train_dataset = ECEDataset(train_data, tokenizer, MAX_LENGTH)
val_dataset = ECEDataset(val_data, tokenizer, MAX_LENGTH)
test_dataset = ECEDataset(test_data, tokenizer, MAX_LENGTH)

print(f"✅ Created PyTorch datasets:")
print(f"   Training:   {len(train_dataset):,} samples")
print(f"   Validation: {len(val_dataset):,} samples")
print(f"   Test:       {len(test_dataset):,} samples")

# Test dataset
sample_item = train_dataset[0]
print(f"\nSample dataset item:")
print(f"  input_ids shape:      {sample_item['input_ids'].shape}")
print(f"  attention_mask shape: {sample_item['attention_mask'].shape}")
print(f"  bio_labels shape:     {sample_item['bio_labels'].shape}")
print(f"  clause_label:         {sample_item['clause_label'].item()}")

### 1.5 Create DataLoaders

Create PyTorch DataLoaders for batching and efficient training.

In [ ]:
# Hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 0  # Set to 0 for compatibility

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"✅ Created DataLoaders:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Training batches:   {len(train_loader):,}")
print(f"   Validation batches: {len(val_loader):,}")
print(f"   Test batches:       {len(test_loader):,}")

# Test a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  input_ids shape:      {sample_batch['input_ids'].shape}")
print(f"  attention_mask shape: {sample_batch['attention_mask'].shape}")
print(f"  bio_labels shape:     {sample_batch['bio_labels'].shape}")
print(f"  clause_label shape:   {sample_batch['clause_label'].shape}")

---
## 🏗️ Part 2: Two-Stage ECE Model Architecture

### 2.1 Define ECE Model

Implement the two-stage ECE model with:
- **Head 1:** Clause Classifier (binary: contains cause?)
- **Head 2:** Span Extractor (BIO tagging)

In [ ]:
class ECEModel(RobertaPreTrainedModel):
    """
    Two-Stage Emotion-Cause-Extraction Model.
    
    Architecture:
        - RoBERTa base model
        - Head 1: Clause classifier (binary)
        - Head 2: Span extractor (BIO tagging)
    """
    
    def __init__(self, config):
        super().__init__(config)
        self.num_bio_labels = len(BIO_LABELS)
        
        # RoBERTa base model
        self.roberta = RobertaModel(config)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
        # Head 1: Clause Classifier (binary: contains cause or not)
        # Uses [CLS] token representation
        self.clause_classifier = nn.Linear(config.hidden_size, 1)
        
        # Head 2: Span Extractor (BIO tagging)
        # Uses all token representations
        self.span_classifier = nn.Linear(config.hidden_size, self.num_bio_labels)
        
        # Initialize weights
        self.post_init()
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        bio_labels=None,
        clause_label=None,
        return_dict=True
    ):
        """
        Forward pass through the model.
        
        Args:
            input_ids: Input token IDs
            attention_mask: Attention mask
            bio_labels: BIO tags for span extraction
            clause_label: Binary label for clause classification
            return_dict: Whether to return dict
            
        Returns:
            Dict with loss and logits
        """
        # Get RoBERTa outputs
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Get representations
        sequence_output = outputs.last_hidden_state  # [batch, seq_len, hidden_size]
        pooled_output = sequence_output[:, 0, :]      # [CLS] token [batch, hidden_size]
        
        # Apply dropout
        sequence_output = self.dropout(sequence_output)
        pooled_output = self.dropout(pooled_output)
        
        # Head 1: Clause Classification
        clause_logits = self.clause_classifier(pooled_output)  # [batch, 1]
        clause_logits = clause_logits.squeeze(-1)               # [batch]
        
        # Head 2: Span Extraction (BIO tagging)
        span_logits = self.span_classifier(sequence_output)     # [batch, seq_len, num_labels]
        
        # Calculate losses if labels provided
        total_loss = None
        clause_loss = None
        span_loss = None
        
        if clause_label is not None and bio_labels is not None:
            # Clause loss (binary classification)
            clause_loss_fct = nn.BCEWithLogitsLoss()
            clause_loss = clause_loss_fct(clause_logits, clause_label)
            
            # Span loss (token classification)
            span_loss_fct = nn.CrossEntropyLoss()
            
            # Flatten for loss calculation
            active_loss = attention_mask.view(-1) == 1
            active_logits = span_logits.view(-1, self.num_bio_labels)[active_loss]
            active_labels = bio_labels.view(-1)[active_loss]
            
            span_loss = span_loss_fct(active_logits, active_labels)
            
            # Combined loss: weighted sum (0.3 * clause + 0.7 * span)
            total_loss = 0.3 * clause_loss + 0.7 * span_loss
        
        if not return_dict:
            output = (clause_logits, span_logits) + outputs[2:]
            return ((total_loss,) + output) if total_loss is not None else output
        
        return {
            'loss': total_loss,
            'clause_loss': clause_loss,
            'span_loss': span_loss,
            'clause_logits': clause_logits,
            'span_logits': span_logits,
            'hidden_states': outputs.hidden_states,
            'attentions': outputs.attentions
        }


# Initialize model
config = RobertaConfig.from_pretrained(MODEL_NAME)
model = ECEModel.from_pretrained(MODEL_NAME, config=config)
model.to(device)

print(f"✅ ECE Model initialized")
print(f"   Base model: {MODEL_NAME}")
print(f"   Number of BIO labels: {len(BIO_LABELS)}")
print(f"   Device: {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Parameters:")
print(f"   Total:     {total_params:,}")
print(f"   Trainable: {trainable_params:,}")

# Test forward pass
model.eval()
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_outputs = model(
        input_ids=test_batch['input_ids'].to(device),
        attention_mask=test_batch['attention_mask'].to(device),
        bio_labels=test_batch['bio_labels'].to(device),
        clause_label=test_batch['clause_label'].to(device)
    )
    
    print(f"\nTest forward pass:")
    print(f"   Loss: {test_outputs['loss'].item():.4f}")
    print(f"   Clause loss: {test_outputs['clause_loss'].item():.4f}")
    print(f"   Span loss: {test_outputs['span_loss'].item():.4f}")
    print(f"   Clause logits shape: {test_outputs['clause_logits'].shape}")
    print(f"   Span logits shape: {test_outputs['span_logits'].shape}")

model.train()
print(f"\n✅ Model ready for training!")

---
## 🚀 Part 3: Training & Evaluation

### 3.1 Initialize Wandb & Training Setup

Configure Weights & Biases for experiment tracking and set up optimizer and scheduler.

In [ ]:
# Training hyperparameters
EPOCHS = 8
LEARNING_RATE = 2e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

# Initialize Weights & Biases
wandb.init(
    project="aura-ece-training",
    name="roberta-base-two-stage-ece",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "loss_weights": {"clause": 0.3, "span": 0.7}
    }
)

print(f"✅ Wandb initialized: {wandb.run.name}")

# Setup optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Calculate total training steps
total_steps = len(train_loader) * EPOCHS

# Setup learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"\n📊 Training Configuration:")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Total steps: {total_steps:,}")
print(f"   Warmup steps: {WARMUP_STEPS}")
print(f"   Optimizer: AdamW")
print(f"   Scheduler: Linear with warmup")

### 3.2 Define Evaluation Function

Implement evaluation function to calculate F1 score using seqeval.

In [ ]:
def evaluate(model, dataloader, device):
    """
    Evaluate the model on a dataset.
    
    Args:
        model: ECE model
        dataloader: DataLoader for evaluation
        device: Device to run on
        
    Returns:
        Dict with evaluation metrics
    """
    model.eval()
    
    total_loss = 0
    total_clause_loss = 0
    total_span_loss = 0
    
    all_predictions = []
    all_labels = []
    all_clause_preds = []
    all_clause_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            # Move to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            bio_labels = batch['bio_labels'].to(device)
            clause_label = batch['clause_label'].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bio_labels=bio_labels,
                clause_label=clause_label
            )
            
            # Accumulate losses
            total_loss += outputs['loss'].item()
            total_clause_loss += outputs['clause_loss'].item()
            total_span_loss += outputs['span_loss'].item()
            
            # Get predictions
            span_preds = torch.argmax(outputs['span_logits'], dim=-1)  # [batch, seq_len]
            clause_preds = torch.sigmoid(outputs['clause_logits']) > 0.5  # [batch]
            
            # Convert to lists for seqeval
            for i in range(input_ids.size(0)):
                # Get active tokens (not padding)
                active_length = attention_mask[i].sum().item()
                
                # Get predictions and labels for this sequence
                pred_labels = [ID2LABEL[p.item()] for p in span_preds[i][:active_length]]
                true_labels = [ID2LABEL[l.item()] for l in bio_labels[i][:active_length]]
                
                all_predictions.append(pred_labels)
                all_labels.append(true_labels)
                
                # Clause predictions
                all_clause_preds.append(clause_preds[i].item())
                all_clause_labels.append(clause_label[i].item())
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_clause_loss = total_clause_loss / len(dataloader)
    avg_span_loss = total_span_loss / len(dataloader)
    
    # Calculate F1 score for BIO tags (using seqeval)
    span_f1 = f1_score(all_labels, all_predictions)
    
    # Calculate clause accuracy
    clause_acc = accuracy_score(all_clause_labels, all_clause_preds)
    
    return {
        'loss': avg_loss,
        'clause_loss': avg_clause_loss,
        'span_loss': avg_span_loss,
        'span_f1': span_f1,
        'clause_accuracy': clause_acc
    }


# Test evaluation function
print("Testing evaluation function...")
eval_metrics = evaluate(model, val_loader, device)

print(f"\n📊 Initial Evaluation (before training):")
print(f"   Loss: {eval_metrics['loss']:.4f}")
print(f"   Span F1: {eval_metrics['span_f1']:.4f}")
print(f"   Clause Accuracy: {eval_metrics['clause_accuracy']:.4f}")
print(f"\n✅ Evaluation function ready!")

### 3.3 Training Loop

Execute the main training loop with validation and model checkpointing.

In [ ]:
# Training tracking
best_f1 = 0.0
best_model_path = Path("best_ece_model.pth")

print(f"{'='*80}")
print(f"STARTING TRAINING")
print(f"{'='*80}\n")

for epoch in range(EPOCHS):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*80}")
    
    # Training phase
    model.train()
    train_loss = 0
    train_clause_loss = 0
    train_span_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bio_labels = batch['bio_labels'].to(device)
        clause_label = batch['clause_label'].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bio_labels=bio_labels,
            clause_label=clause_label
        )
        
        loss = outputs['loss']
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        scheduler.step()
        
        # Accumulate losses
        train_loss += loss.item()
        train_clause_loss += outputs['clause_loss'].item()
        train_span_loss += outputs['span_loss'].item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
        
        # Log to wandb every 50 steps
        if (batch_idx + 1) % 50 == 0:
            wandb.log({
                'train/loss': loss.item(),
                'train/clause_loss': outputs['clause_loss'].item(),
                'train/span_loss': outputs['span_loss'].item(),
                'train/learning_rate': scheduler.get_last_lr()[0],
                'train/step': epoch * len(train_loader) + batch_idx
            })
    
    # Calculate average training losses
    avg_train_loss = train_loss / len(train_loader)
    avg_train_clause = train_clause_loss / len(train_loader)
    avg_train_span = train_span_loss / len(train_loader)
    
    print(f"\n📊 Training Results:")
    print(f"   Average Loss:        {avg_train_loss:.4f}")
    print(f"   Average Clause Loss: {avg_train_clause:.4f}")
    print(f"   Average Span Loss:   {avg_train_span:.4f}")
    
    # Validation phase
    print(f"\n🔍 Running validation...")
    val_metrics = evaluate(model, val_loader, device)
    
    print(f"\n📊 Validation Results:")
    print(f"   Loss:            {val_metrics['loss']:.4f}")
    print(f"   Span F1:         {val_metrics['span_f1']:.4f}")
    print(f"   Clause Accuracy: {val_metrics['clause_accuracy']:.4f}")
    
    # Log epoch metrics to wandb
    wandb.log({
        'epoch': epoch + 1,
        'train/epoch_loss': avg_train_loss,
        'train/epoch_clause_loss': avg_train_clause,
        'train/epoch_span_loss': avg_train_span,
        'val/loss': val_metrics['loss'],
        'val/span_f1': val_metrics['span_f1'],
        'val/clause_accuracy': val_metrics['clause_accuracy']
    })
    
    # Save best model based on F1 score
    if val_metrics['span_f1'] > best_f1:
        best_f1 = val_metrics['span_f1']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_f1': best_f1,
            'val_metrics': val_metrics
        }, best_model_path)
        
        print(f"\n✅ New best model saved! F1: {best_f1:.4f}")
        wandb.run.summary["best_f1"] = best_f1
        wandb.run.summary["best_epoch"] = epoch + 1
    
    print(f"\n{'='*80}")

print(f"\n{'='*80}")
print(f"TRAINING COMPLETE!")
print(f"{'='*80}")
print(f"Best F1 Score: {best_f1:.4f}")
print(f"Best model saved to: {best_model_path}")
print(f"{'='*80}\n")

wandb.finish()

---
## 🔍 Part 4: Error Analysis & Testing

### 4.1 Load Best Model

Load the best saved model checkpoint for final evaluation.

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"✅ Loaded best model from: {best_model_path}")
print(f"   Best F1 Score: {checkpoint['best_f1']:.4f}")
print(f"   Epoch: {checkpoint['epoch']}")
print(f"\n📊 Validation Metrics at Best Checkpoint:")
for metric, value in checkpoint['val_metrics'].items():
    print(f"   {metric}: {value:.4f}")

### 4.2 Final Test Set Evaluation

Evaluate the best model on the held-out test set.

In [ ]:
print(f"{'='*80}")
print(f"FINAL TEST SET EVALUATION")
print(f"{'='*80}\n")

test_metrics = evaluate(model, test_loader, device)

print(f"\n📊 Test Set Results:")
print(f"   Loss:            {test_metrics['loss']:.4f}")
print(f"   Span F1:         {test_metrics['span_f1']:.4f}")
print(f"   Clause Accuracy: {test_metrics['clause_accuracy']:.4f}")

print(f"\n{'='*80}")

### 4.3 Error Analysis on Test Examples

Run inference on 20 test examples and analyze model predictions vs ground truth.

In [ ]:
def extract_cause_from_bio(tokens: List[str], bio_tags: List[str]) -> str:
    """Extract cause text from BIO tags."""
    cause_tokens = []
    for token, tag in zip(tokens, bio_tags):
        if tag.startswith('B-') or tag.startswith('I-'):
            cause_tokens.append(token)
    return ' '.join(cause_tokens) if cause_tokens else "[No cause detected]"

def analyze_predictions(model, dataset, tokenizer, num_examples=20, device='cpu'):
    """Analyze model predictions on test examples."""
    model.eval()
    
    print(f"{'='*80}")
    print(f"ERROR ANALYSIS: {num_examples} Test Examples")
    print(f"{'='*80}\n")
    
    # Sample random examples
    indices = np.random.choice(len(dataset), min(num_examples, len(dataset)), replace=False)
    
    correct_count = 0
    
    for idx, i in enumerate(indices):
        example = dataset[i]
        
        # Get original data
        original_text = test_data[i]['text']
        true_cause = test_data[i]['cause']
        emotion = test_data[i]['emotion']
        
        # Prepare batch
        input_ids = example['input_ids'].unsqueeze(0).to(device)
        attention_mask = example['attention_mask'].unsqueeze(0).to(device)
        true_bio_labels = example['bio_labels']
        
        # Get predictions
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            # Get predicted BIO tags
            span_logits = outputs['span_logits']  # [1, seq_len, num_labels]
            predicted_tags_ids = torch.argmax(span_logits, dim=-1)[0]  # [seq_len]
            
            # Get clause prediction
            clause_logit = outputs['clause_logits']  # [1, 1]
            clause_pred = torch.sigmoid(clause_logit).item()
        
        # Convert to tokens and tags
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy())
        
        # Convert predicted tag IDs to BIO labels
        id2label = {0: 'O', 1: 'B-CAUSE', 2: 'I-CAUSE'}
        predicted_bio = [id2label[tag_id.item()] for tag_id in predicted_tags_ids]
        true_bio = [id2label[label_id] for label_id in true_bio_labels.cpu().numpy()]
        
        # Extract causes
        predicted_cause = extract_cause_from_bio(tokens, predicted_bio)
        true_cause_from_bio = extract_cause_from_bio(tokens, true_bio)
        
        # Clean up tokens (remove special tokens)
        predicted_cause = predicted_cause.replace('<s>', '').replace('</s>', '').replace('Ġ', ' ').strip()
        
        # Check if prediction is correct (simple string match)
        is_correct = predicted_cause.lower() in true_cause.lower() and predicted_cause != "[No cause detected]"
        if is_correct:
            correct_count += 1
        
        # Print analysis
        print(f"{'─'*80}")
        print(f"Example {idx + 1}/{num_examples} {'✅ CORRECT' if is_correct else '❌ INCORRECT'}")
        print(f"{'─'*80}")
        print(f"Text:     {original_text[:100]}...")
        print(f"Emotion:  {emotion}")
        print(f"True Cause:      {true_cause}")
        print(f"Predicted Cause: {predicted_cause}")
        print(f"Clause Score:    {clause_pred:.3f} ({'HAS_CAUSE' if clause_pred > 0.5 else 'NO_CAUSE'})")
        print()
    
    accuracy = correct_count / num_examples
    print(f"{'='*80}")
    print(f"Error Analysis Summary:")
    print(f"  Correct:   {correct_count}/{num_examples}")
    print(f"  Accuracy:  {accuracy:.2%}")
    print(f"{'='*80}\n")
    
    return accuracy

# Run error analysis
error_accuracy = analyze_predictions(
    model=model,
    dataset=test_dataset,
    tokenizer=tokenizer,
    num_examples=20,
    device=device
)

---
## ✅ Week 9 ECE Model Training Complete!

**Summary:**
- ✅ Two-stage ECE model trained with RoBERTa
- ✅ Clause classifier + BIO tagger for span extraction
- ✅ Combined loss function (weighted)
- ✅ Evaluation with seqeval F1 score
- ✅ Model checkpointing and best model saved
- ✅ Test set evaluation completed
- ✅ Error analysis on 20 examples

**Next Steps:**
1. Fine-tune hyperparameters (learning rate, loss weights)
2. Experiment with different transformer models (BERT, ELECTRA)
3. Add data augmentation techniques
4. Integrate model into Aura backend API
5. Create inference endpoint for real-time ECE

**Model Artifacts:**
- Best model checkpoint: `models/ece_roberta_best.pt`
- Training logs: Weights & Biases dashboard

---